In [1]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
from google.colab import files
spark = SparkSession.builder.appName('SalesAnalysis').getOrCreate()

#browse and select multiple csv files
uploaded = files.upload()

#load selected csv files into pyspark dataframes
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records: ",df.count)
    df.show(5)

Saving lazy_execution_products (1).csv to lazy_execution_products (1) (2).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1) (1).csv
Saving lazy_execution_stores (1).csv to lazy_execution_stores (1) (1).csv

File: lazy_execution_products (1) (2).csv
Records:  <bound method DataFrame.count of DataFrame[product_id: string, product_name: string, category: string, list_price: int, cost_price: int]>
+----------+-------------------+-----------+----------+----------+
|product_id|       product_name|   category|list_price|cost_price|
+----------+-------------------+-----------+----------+----------+
|      P001|     Wireless Mouse|Accessories|       799|       420|
|      P002|Mechanical Keyboard|Accessories|      2999|      1600|
|      P003|          USB-C Hub|Accessories|      1899|       920|
|      P004|    27-inch Monitor|   Displays|     16999|     11400|
|      P005|          Webcam HD|Accessories|      2499|      1400|
+----------+-------------------+-----

In [7]:
from pyspark.sql.functions import col

In [9]:
# Identify the sales dataframe from the loaded datasets
sales_file_key = [k for k in dataframes.keys() if 'sales' in k][0]
sales_df = dataframes[sales_file_key]

#Transformation
completed_df = sales_df.filter(
    col('order_status') == "Completed"
)
print("Transformation created")

#Action 1
print("Completed Orders:", completed_df.count())

#Action 2
completed_df.show(5)

Transformation created
Completed Orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
+--------+-------

In [10]:
#Cache the filtering dataframe
completed_df.cache()

print("IS Cached:", completed_df.is_cached)

#first action
completed_df.count()

#resue cached data
completed_df.show(5)

#Another action
completed_df.groupBy('product_id').count().show()

IS Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
+--------+----------+--------+----------+-------

In [11]:
from pyspark import StorageLevel
#create filter dataframe
high_quantity_df = sales_df.filter(
    col('quantity') > 5
)

#persist using memory and disc
high_quantity_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

#trigger execution
high_quantity_df.count()

#reuse persist data
high_quantity_df.show(5)

#check storage level
print(high_quantity_df.storageLevel)



+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
+--------+----------+--------+----------+--------+----------+---

In [12]:
#check cache status
print("Before:",completed_df.is_cached)

#unpersist
completed_df.unpersist(blocking=True)

#check again
print("After:",completed_df.is_cached)

#remove the other persisted dataframe
high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [14]:
from pyspark.sql import functions as F

# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [16]:
import time

# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 1.083 seconds
Second Execution: 0.09 seconds


In [18]:
#partition data by store_id
store_partitioned_df = sales_df.repartition(
    8,
    "store_id"
)
print("Partitions:",
      store_partitioned_df.rdd.getNumPartitions())

#Inspect partitions distribution
store_partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id","store_id"
).pivot(
    "partition_id"
).count().orderBy(
    "partition_id","store_id"
).show(30)


Partitions: 8
+------------+--------+----+----+----+----+----+
|partition_id|store_id|   0|   1|   3|   4|   5|
+------------+--------+----+----+----+----+----+
|           0|     S01|3180|NULL|NULL|NULL|NULL|
|           1|     S02|NULL|3121|NULL|NULL|NULL|
|           1|     S08|NULL|3122|NULL|NULL|NULL|
|           3|     S06|NULL|NULL|3033|NULL|NULL|
|           4|     S05|NULL|NULL|NULL|3117|NULL|
|           4|     S07|NULL|NULL|NULL|3209|NULL|
|           5|     S03|NULL|NULL|NULL|NULL|3132|
|           5|     S04|NULL|NULL|NULL|NULL|3086|
+------------+--------+----+----+----+----+----+



In [21]:
#create 8 partitions
df8 = sales_df.repartition(8)

#reduce using repartition
df_repartition = df8.repartition(3)

df_coalesce = df8.coalesce(3)

print("Repartition:", df_repartition.rdd.getNumPartitions())

print("Coalesce:", df_coalesce.rdd.getNumPartitions())

#example execution plans
print("Repartition")
df_repartition.explain()

print("Coalesce")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
Repartition
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1049]
         +- FileScan csv [order_id#157,order_date#158,store_id#159,product_id#160,quantity#161,unit_price#162,discount_pct#163,payment_method#164,order_status#165,sales_channel#166] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (1) (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1049]
   +- FileScan csv [order_id#157,order_date#158,store_id#159,product_id#160,quantity#161,unit_price#162,discount_pct#163,payment_method#164,order_status#165,sales_channel#166] Batched: fal

In [23]:
df_repartition.rdd.getNumPartitions()

3

In [24]:
df_12 = sales_df.repartition(12)
df_12.rdd.getNumPartitions()

12

In [26]:
df_12.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



In [28]:
df_repartitioned = sales_df.repartition(6, "product_id")

In [30]:
df_repartitioned_coalesce = df_repartitioned.coalesce(4)
df_repartitioned_coalesce.rdd.getNumPartitions()

4